# Bronze Layer: Holidays API (Simple Overwrite)

**Source**: Nager.Date Public Holidays API  

This notebook fetches holiday data and overwrites the target table. Simple mode for Free Edition.

In [0]:
import requests
from pyspark.sql.functions import current_timestamp

def fetch_holidays(years, countries):
    all_holidays = []
    for country in countries:
        for year in years:
            url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country}"
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200:
                data = resp.json()
                for h in data:
                    all_holidays.append({
                        "country_code": country,
                        "date": h.get("date"),
                        "name": h.get("name"),
                        "local_name": h.get("localName"),
                        "is_global": h.get("global")
                    })
    return all_holidays

years = [2010, 2011, 2012, 2013, 2014]
countries = ["US", "GB", "DE", "FR", "CA", "AU"]

print("🌐 Fetching holidays from API...")
data = fetch_holidays(years, countries)
df = spark.createDataFrame(data).withColumn("ingestion_timestamp", current_timestamp())

print("Saving to Bronze (Overwrite)...")
df.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.holidays_api_cdc")
print("✓ Done.")